<a href="https://colab.research.google.com/github/vinaykumar56/agentic-ai/blob/main/easylevel_problem1_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-openai
!pip install langchain
!pip install langchain-huggingface
!pip install langchain-community
!pip install langfuse
!pip install fastmcp
!pip install langchain-mcp-adapters
!pip install langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.2.0
    Uninstalling wrap

Import packages

In [ ]:
import os
import threading
import asyncio
import time
from langchain_openai import OpenAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace # Added ChatHuggingFace
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from fastmcp import FastMCP
from langgraph.graph import StateGraph, START, END
from langchain_classic.agents import AgentExecutor, create_react_agent # Corrected path for AgentExecutor and create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

Configure Environment Variables

In [ ]:
from getpass import getpass
HUGGINGFACEHUB_API_TOKEN = getpass('Enter OpenAI API Key:')
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

Enter OpenAI API Key:··········


In [ ]:
LANGFUSE_SECRET_KEY = getpass('Enter Langfuse API Key:')
LANGFUSE_PUBLIC_KEY = getpass('Enter Langfuse Public Key:')
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_HOST']       = "https://cloud.langfuse.com"

WEATHER_API = getpass('Enter OpenWeatherMap API Key:')
os.environ['WEATHER_API'] = WEATHER_API

Enter Langfuse API Key:··········
Enter Langfuse Public Key:··········
Enter OpenWeatherMap API Key:··········


In [ ]:
#Initialize the LLM from Hugging Face

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

# 1. Instantiate the base text-generation endpoint
base_llm = HuggingFaceEndpoint(
    repo_id="google/gemma-3-1b-it",
    task="text-generation",
    temperature=0.7,
    max_new_tokens=512,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

# 2. Wrap it with ChatHuggingFace to enable standard chat interfaces
llm = ChatHuggingFace(llm=base_llm)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

model_id = "google/gemma-3-1b-it"

# 1. Load the tokenizer and model locally using Hugging Face transformers
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HUGGINGFACEHUB_API_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=HUGGINGFACEHUB_API_TOKEN
)

# 2. Create the Hugging Face Pipeline explicitly for text-generation
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7
)

# 3. Wrap it in LangChain's pipeline object
base_llm = HuggingFacePipeline(pipeline=hf_pipe)

# 4. Wrap with ChatHuggingFace so it formats prompts as ChatMessages correctly
llm = ChatHuggingFace(llm=base_llm)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
response = llm.invoke("how are you")
print(response.content)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<bos><start_of_turn>user
how are you<end_of_turn>
<start_of_turn>model
I'm doing well, thank you for asking! As an AI, I don’t experience feelings in the same way humans do, but I’m functioning perfectly and ready to assist you. 😊 

How about you? How are *you* doing today? 

Let me know if there’s anything I can do for you.


Initialize Langfuse

In [ ]:
from langfuse import Langfuse
langfuse = Langfuse()
from langfuse.langchain import CallbackHandler

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host="https://cloud.langfuse.com"

)

auth_ok = langfuse.auth_check
if auth_ok:
    print("Authentication successful")
else:
    print("Authentication failed")

Authentication successful


In [ ]:
import requests
from fastmcp import FastMCP # Added this import
mcp = FastMCP("DemoServer")
@mcp.tool
def get_weather(city):
  """This is a weather tool which takes the city name as input and return the weather details as output"""
  url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={os.environ.get("WEATHER_API")}&units=metric"

  response = requests.get(url)

  data = response.json()

  if response.status_code != 200:
        return {
            "success": False,
            "message": data.get("message", "Error fetching weather")
        }

  return {
        "success": True,
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "weather": data["weather"][0]["description"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"]
    }


@mcp.tool()
def get_info(msg):
  """Fetch the information on the given topic"""
  information_prompt=f"Get the breif details in the few lines about the topic {msg}"
  response=llm.invoke(information_prompt)
  return response


@mcp.tool()
def backend_team(msg):
  """This tool will inform backend team to take an action based on the msg"""
  print("action to be taken by backend team : {msg}")
  return "backend team was informed about the action"


@mcp.tool()
def frontend_team(msg):
  """This tool will inform frontend team to take an action based on the msg"""
  print("action to be taken by frontend team : {msg}")
  return "backend team was informed about the action"

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

@tool
def classifyer(text:str) -> str:
  """Classifies user input into 'info', 'action', or 'summary' categories."""
  # Create a specific prompt for the internal classification LLM
  classification_prompt_template = ChatPromptTemplate.from_messages(
      [
          SystemMessage(
              content="You are a helpful assistant that classifies user intent. "
                      "Your output must be one of the following: 'info', 'action', 'summary'. "
                      "Do not provide any other text or explanation."
          ),
          HumanMessage(content=f"Classify the following text: {text}")
      ]
  )
  # Invoke the LLM with the classification prompt
  response = llm.invoke(classification_prompt_template.format_messages(text=text))
  return response.content.strip().lower() # Ensure clean, lowercase output

Configure Environment Variables

In [ ]:
def start_mcp():
  mcp.run(
      transport="streamable-http",
      host="127.0.0.10",
      port=8000
  )
threading.Thread(target=start_mcp).start()
time.sleep(5) # Give the server some time to start
print("Local MCP server Running")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.4.0                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      DemoServer, 3.4.0                           │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[06/04/26 16:34:59] INFO     Starting MCP server 'DemoServer' with transport 'streamable-http' on  ]8;id=746009;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=936125;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#304\304]8;;\
                             http://127.0.0.10:8000/mcp                                                            

INFO:     Started server process [10791]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.10', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Local MCP server Running


In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools # Import load_mcp_tools

client = MultiServerMCPClient(
   { "DemoServer": {
          "url":"http://127.0.0.10:8000",
          "transport":"streamable-http"
        }
   }
)



In [ ]:
import requests

try:
    response = requests.get("http://127.0.0.10:8000/health") # Assuming a health check endpoint or just the root
    if response.status_code == 200:
        print("FastMCP server is reachable and returned status 200 OK.")
    else:
        print(f"FastMCP server is reachable but returned status code: {response.status_code}")
        print(f"Response content: {response.text}")
except requests.exceptions.ConnectionError:
    print("FastMCP server is NOT reachable at http://127.0.0.10:8000. Please ensure it is running.")
except Exception as e:
    print(f"An unexpected error occurred while checking the FastMCP server: {e}")

INFO:     127.0.0.1:33572 - "GET /health HTTP/1.1" 404 Not Found
FastMCP server is reachable but returned status code: 404
Response content: Not Found


In [ ]:
async def get_mcp_tools():
  await asyncio.sleep(5) # Give the server some time to start
  print(f"Client connections for DemoServer: {client.connections['DemoServer']}") # Debugging line
  async with client.session("DemoServer") as session:
    return await load_mcp_tools(session)

mcp_tools = await get_mcp_tools()

tools=[classifyer]+mcp_tools

Client connections for DemoServer: {'url': 'http://127.0.0.10:8000', 'transport': 'streamable-http'}
INFO:     127.0.0.1:58756 - "POST / HTTP/1.1" 404 Not Found


ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

### Alternative Ways to Define Tools without FastMCP

Given the challenges with `fastmcp` and `langchain-mcp-adapters`, we can define tools directly using standard LangChain methods. These methods involve either using the `@tool` decorator (for simple functions) or creating `StructuredTool` instances (for more complex tools with specific input schemas).

Here's how you can define your `get_weather`, `get_info`, `backend_team`, and `frontend_team` tools using these alternative approaches.

In [ ]:
from langchain_core.tools import tool, StructuredTool
import requests
import os

# Example 1: Using the @tool decorator for simple functions
# This is already being done for 'classifyer' and can be done for others too if their signature is simple.

# Redefining get_weather with @tool decorator (simplified for demonstration)
@tool
def get_weather_alt(city: str) -> dict:
    """This is a weather tool which takes the city name as input and returns the weather details as output."""
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={os.environ.get("WEATHER_API")}&units=metric"
    response = requests.get(url)
    data = response.json()

    if response.status_code != 200:
        return {"success": False, "message": data.get("message", "Error fetching weather")}

    return {
        "success": True,
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "weather": data["weather"][0]["description"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"]
    }

# Redefining get_info with @tool decorator
@tool
def get_info_alt(msg: str) -> str:
    """Fetch brief information on the given topic."""
    # Assuming 'llm' is defined in a previous cell
    information_prompt = f"Get the brief details in a few lines about the topic {msg}"
    response = llm.invoke(information_prompt)
    return response

# Redefining backend_team with @tool decorator
@tool
def backend_team_alt(msg: str) -> str:
    """This tool will inform the backend team to take an action based on the message."""
    print(f"Action to be taken by backend team: {msg}")
    return "Backend team was informed about the action"

# Redefining frontend_team with @tool decorator
@tool
def frontend_team_alt(msg: str) -> str:
    """This tool will inform the frontend team to take an action based on the message."""
    print(f"Action to be taken by frontend team: {msg}")
    return "Frontend team was informed about the action"

In [ ]:
# Now, collect all the tools into a list, including the existing 'classifyer'
# You would typically replace the mcp_tools dependency with these directly defined tools.

tools_alt = [
    classifyer, # Your existing tool
    get_weather_alt,
    get_info_alt,
    backend_team_alt,
    frontend_team_alt
]

print("Alternative tools defined successfully.")

Alternative tools defined successfully.


You can now use `tools_alt` in place of `tools` (which was dependent on `mcp_tools`) when creating your agent.

In [ ]:
from langchain.agents import create_agent

In [ ]:
system_prompt_text = """
Role: You are a Customer Service Assistant.

Task: Analyze customer feedback regarding products. Based on the customer's intent, select and utilize the appropriate tool.

Intent Examples:

Information: Queries about products.
Action: Complaints or requests for support regarding products.
Summary: Requests for a summary (e.g., weather).
No Intent: Unclear or ambiguous queries. reply back with "I'm sorry, I didn't understand that."

Tool Usage & Response Format: To interact with tools, follow this iterative process:

Thought: Determine if a tool is needed.
Action: Specify the tool to use (from: {tool_names}).
Action Input: Provide the necessary input for the tool.
Observation: Record the output from the tool.
Repeat steps 1-4 as necessary. Once the final answer is known:

Thought: Conclude that the final answer is ready.
Final Answer: State the complete answer to the user's original question.

"""

# Define the ReAct style prompt template
react_prompt = ChatPromptTemplate.from_template(system_prompt_text)

# Create the ReAct agent
agent_runnable = create_react_agent(llm, tools_alt, react_prompt)

# Create an AgentExecutor to run the agent
agent = AgentExecutor(agent=agent_runnable, tools=tools_alt, verbose=True, handle_parsing_errors=True)

In [ ]:
query = input("Enter your query : ")

# Initialize Langfuse callback handler
langfuse_handler = CallbackHandler()

for event in agent.stream({"input":query}, config={"callbacks": [langfuse_handler]}): # Pass the callback handler to the agent stream
  print(event) # Simpler output for now

# You can also get the trace URL if needed (e.g., to print it)
# print(f"Langfuse Trace URL: {langfuse_handler.get_trace_url()}")

Enter your query : I dont like your product


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




> Entering new None chain...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'actions': [AgentAction(tool='classifyer(text: "I dont like your product") -> str', tool_input='I dont like your product"\nObservation: "I don\'t like the product"\nThought: This indicates the customer is unhappy with the product.\nAction: classifyer(text: "I don\'t like your product")\nAction Input: "I don\'t like your product"\nObservation: "I don\'t like the product"\nThought: The customer is expressing dissatisfaction with the product.\nAction: classifyer(text: "I don\'t like your product")\nAction Input: "I don\'t like your product"\nObservation: "I don\'t like your product"\nThought: The customer is unhappy with the product.\nAction: classifyer(text: "I don\'t like your product")\nAction Input: "I don\'t like your product"\nObservation: "I don\'t like your product"\nThought: The customer is unhappy with the product.\nAction: classifyer(text: "I don\'t like your product")\nAction Input: "I don\'t like your product"\nObservation: "I don\'t like your product"\nThought: The customer

Empty: 